# H-mode global confinement $\tau_E$ scaling
Reproduces the basic engineering-variable results of Verdoolaege *et al.* 2021, Nucl. Fusion **61** 076006 (DB5.2.3) from the IMAS-migrated H-mode database.

In [ ]:
import os
import numpy as np
import imas

ROOT = os.path.dirname(os.getcwd())
HMODE_DIR = os.path.join(ROOT, "resources", "results", "hmode")

pulse_dirs = sorted(
    os.path.join(HMODE_DIR, d)
    for d in os.listdir(HMODE_DIR)
    if d.startswith("pulse_")
)
N = len(pulse_dirs)
print(f"{N} pulses in {HMODE_DIR}")

## Scaling variables

| Variable | IMAS path | Unit |
|---|---|---|
| $\tau_{E,th}$ (TAUTH) | `summary/global_quantities/tau_energy/value` | s |
| $I_p$ | `summary/global_quantities/ip/value` | A |
| $B_t$ | `summary/global_quantities/b0/value` | T |
| $\bar n_e$ | `summary/line_average/n_e/value` | m^-3 |
| $P_{l,th}$ | `summary/global_quantities/power_loss/value` | W |
| $R_{geo}$ | `summary/global_quantities/r0/value` | m |
| $V$ | `summary/global_quantities/volume/value` | m^3 |
| $a$ | `summary/boundary/minor_radius/value` | m |
| $M_{eff}$ | `summary/volume_average/meff_hydrogenic/value` | AMU |
| $\delta$ | `equilibrium/time_slice(0)/boundary/triangularity` | — |
| TOK | `summary/machine` | — |
| PHASE | `temporary/constant_string0d` (by `identifier.name`) | — |
| SELDB5 | `temporary/constant_integer0d` (by `identifier.name`) | — |

Derived (paper definitions): $\kappa_a = V/(2\pi R_{geo}\,\pi a^2)$, $\epsilon = a/R_{geo}$.

In [ ]:
def first_scalar(arr):
    """Return first element of array (NaN if empty/unavailable)."""
    try:
        a = np.asarray(arr, dtype=float)
        return float(a.flat[0]) if a.size else np.nan
    except Exception:
        return np.nan


def temp_by_name(bucket, name):
    """Read a temporary-IDS slot by its identifier.name (robust to slot index)."""
    for el in bucket:
        try:
            if str(el.identifier.name).strip() == name:
                return el.value
        except Exception:
            continue
    return None


TAU   = np.full(N, np.nan)          # thermal energy confinement time [s]
IP    = np.full(N, np.nan)          # plasma current [A]
BT    = np.full(N, np.nan)          # vacuum B_t at R0 [T]
NEL   = np.full(N, np.nan)          # line-averaged n_e [m^-3]
PLTH  = np.full(N, np.nan)          # thermal loss power [W]
RGEO  = np.full(N, np.nan)          # geometric major radius [m]
VOL   = np.full(N, np.nan)          # plasma volume [m^3]
AMIN  = np.full(N, np.nan)          # minor radius [m]
MEFF  = np.full(N, np.nan)          # effective hydrogenic mass [AMU] (~PGASA)
DELTA = np.full(N, np.nan)          # triangularity
TOK   = np.empty(N, dtype=object)   # tokamak name
PHASE = np.empty(N, dtype=object)   # discharge phase (ELM type)
SELDB5 = np.full(N, np.nan)         # DB5 standard-selection flag

for i, pulse_dir in enumerate(pulse_dirs):
    uri = f"imas:hdf5?path={pulse_dir};pulse=0"
    if i % 100 == 0:
        print(f"Processing pulse {i}/{N}")
    with imas.DBEntry(uri, "r") as entry:
        s = entry.get("summary", lazy=True)
        TAU[i]   = first_scalar(s.global_quantities.tau_energy.value)
        IP[i]    = first_scalar(s.global_quantities.ip.value)
        BT[i]    = np.abs(first_scalar(s.global_quantities.b0.value))
        NEL[i]   = first_scalar(s.line_average.n_e.value)
        PLTH[i]  = first_scalar(s.global_quantities.power_loss.value)
        RGEO[i]  = first_scalar(s.global_quantities.r0.value)
        VOL[i]   = first_scalar(s.global_quantities.volume.value)
        AMIN[i]  = first_scalar(s.boundary.minor_radius.value)
        MEFF[i]  = first_scalar(s.volume_average.meff_hydrogenic.value)
        TOK[i]   = str(s.machine).strip()

        eq = entry.get("equilibrium", lazy=True)
        DELTA[i] = first_scalar(eq.time_slice[0].boundary.triangularity)

        tmp = entry.get("temporary", lazy=True)
        ph = temp_by_name(tmp.constant_string0d, "PHASE")
        PHASE[i] = str(ph).strip() if ph is not None else ""
        SELDB5[i] = first_scalar(temp_by_name(tmp.constant_integer0d, "SELDB5"))

print("Done.")

In [ ]:
# Units and derived variables (formula in paper)
tau_s    = TAU                                 # [s]
ip_ma    = np.abs(IP) / 1e6                     # [MA]
Bt_T     = np.abs(BT)                           # [T]
ne_19    = NEL / 1e19                           # [10^19 m^-3]
Ploss_MW = PLTH / 1e6                           # [MW]
kappa_a  = VOL / (2.0 * np.pi * RGEO * np.pi * AMIN**2)   # paper kappa_a
eps      = AMIN / RGEO                          # inverse aspect ratio
one_delta = 1.0 + DELTA                         # 1 + delta

# Subset selection: STD5 standard set, ELMy H-mode
PHASE_str = PHASE.astype(str)
std5 = (SELDB5 == 1)
elmy = np.char.startswith(PHASE_str, "HG") | np.char.startswith(PHASE_str, "HS")

# Check if theres a difference?
print(f"Is std5 == elmy?: {np.array_equal(std5, elmy)}")

if not np.array_equal(std5,elmy):
    print(f"No, they differ in: {np.sum(std5 != elmy)} places")
    

predictors = [tau_s, ip_ma, Bt_T, ne_19, Ploss_MW, RGEO, kappa_a, eps, MEFF]
finite = np.all([np.isfinite(p) for p in predictors], axis=0)
positive = (tau_s > 0) & (Ploss_MW > 0) & (ip_ma > 0) & (Bt_T > 0) & (ne_19 > 0)
sel = std5 & elmy & finite & positive

print(f"Total pulses     : {N}")
print(f"SELDB5 == 1      : {np.sum(std5)}")
print(f"ELMy H (HG/HS)   : {np.sum(elmy)}")
print(f"STD5 ELMy + valid: {np.sum(sel)}")
print("\nper machine (STD5 ELMy, valid):")
for tok in np.unique(TOK[sel]):
    print(f"  {tok:10s}: {np.sum((TOK == tok) & sel)}")

## Table 2 — engineering-variable ranges (STD5 ELMy H)
Compare with paper Table 2 (DB5.2.3-STD5 ELMy H). 

In [ ]:
cols = {
    "tau_E,th [s]": tau_s[sel],
    "Ip [MA]":      ip_ma[sel],
    "Bt [T]":       Bt_T[sel],
    "ne [1e19]":    ne_19[sel],
    "Pl,th [MW]":   Ploss_MW[sel],
    "Rgeo [m]":     RGEO[sel],
    "1+delta":      one_delta[sel],
    "kappa_a":      kappa_a[sel],
    "eps":          eps[sel],
    "Meff":         MEFF[sel],
}
print(f"{'variable':14s} {'min':>9s} {'max':>9s} {'mean':>9s} {'median':>9s} {'std':>9s}")
for name, x in cols.items():
    print(f"{name:14s} {np.min(x):9.4g} {np.max(x):9.4g} {np.mean(x):9.4g} {np.median(x):9.4g} {np.std(x):9.4g}")

## Minimal engineering scaling (OLS in log space)
Power law $\tau_{E,th} = \alpha_0\, I_p^{\alpha_I} B_t^{\alpha_B} \bar n_e^{\alpha_n} P_{l,th}^{\alpha_P} R_{geo}^{\alpha_R} \kappa_a^{\alpha_\kappa} \epsilon^{\alpha_\epsilon} M_{eff}^{\alpha_M}$ (paper eq. 2), fitted by ordinary least squares on $\ln$-transformed data. This is the unweighted analog of the paper's WLS (Table 9 'ELMy H'); compare to IPB98(y,2) (Table 7).

In [ ]:
# Design matrix in log space (Ip in MA, ne in 1e19, Pl,th in MW — absorbed into intercept)
y = np.log(tau_s[sel])
regressors = {
    "ln Ip":   np.log(ip_ma[sel]),
    "ln Bt":   np.log(Bt_T[sel]),
    "ln ne":   np.log(ne_19[sel]),
    "ln Plth": np.log(Ploss_MW[sel]),
    "ln Rgeo": np.log(RGEO[sel]),
    "ln kapa": np.log(kappa_a[sel]),
    "ln eps":  np.log(eps[sel]),
    "ln Meff": np.log(MEFF[sel]),
}
X = np.column_stack([np.ones_like(y)] + list(regressors.values()))
coef, *_ = np.linalg.lstsq(X, y, rcond=None)

names = ["ln alpha0"] + list(regressors.keys())
# IPB98(y,2) reference exponents (Table 7): aI,aB,an,aP,aR,akappa,aeps,aM
ipb98 = {"ln Ip":0.93, "ln Bt":0.15, "ln ne":0.41, "ln Plth":-0.69,
         "ln Rgeo":1.97, "ln kapa":0.78, "ln eps":0.58, "ln Meff":0.19}

print(f"{'param':10s} {'this OLS':>10s} {'IPB98(y,2)':>12s}")
print(f"{'alpha0':10s} {np.exp(coef[0]):>10.4g} {0.0562:>12.4g}")
for nm, c in zip(names[1:], coef[1:]):
    ref = ipb98.get(nm, np.nan)
    print(f"{nm:10s} {c:>10.3f} {ref:>12.3f}")

resid = y - X @ coef
rmse = np.sqrt(np.mean(resid**2))
r2 = 1.0 - np.sum(resid**2) / np.sum((y - y.mean())**2)
print(f"\nN = {sel.sum()}   RMSE(log) = {rmse:.3f}   R^2 = {r2:.3f}")